# prep.metadata

Although it might not be the right term, we call "metadata" here to the data about site, location, habitat, etc that is fundamental for our study but does not come from sequencing and taxonomic profiling pipelines. We use a few different files from McLeish et al. 2024 to generate a single metadata file that contains in each row: 

- A site code 
- A library code
- Habitat
- Number of samples
- Host taxon (scientific name)

In [1]:
import pandas as pd
import seaborn as sns 
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
db.toc()

┌──────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│         name         │                                                                                  description                                                                                   │
│       varchar        │                                                                                    varchar                                                                                     │
├──────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ D_sites              │ This table contains key information about each of the libraries, such as their site, habitat and host                                                                  

## Load sample and library data

The study metadata is contained in two files:
- S2: it contains information about libraries, such as their collection code and the number of extracts. It also includes the habitat associated to the site of each library, which is a bit strange, because it should not be located here.
- S1: it contains site data, except the habitat of each site.


We will just read the two files, create a library-collection code-site index, and then merge the files. 


We are not computing host data, as we are not using it right now.

In [2]:
libraries = pd.read_csv("input/mcleish24.TableS2.csv", sep=';')
libraries

,Library_code,Host_taxon,Habitat,Collection_code,No_extracts
0,PV001,Amaranthus sp,Crop,M1V,8
1,PV002,Convolvulus arvensis,Crop,M1V,11
2,PV003,Cucumis melo,Crop,M1V,13
3,PV003bgi,Cucumis melo,Crop,M1V,13
4,PV004bgi,Cyperus longus,Crop,M1V,7
...,...,...,...,...,...
318,PV586,Fumaria parviflora,Crop,H1P,8
319,PV587,Hirschfeldia incana,Crop,H1P,8
320,PV588,Hordeum vulgare,Crop,H1P,8
321,PV589,Hordeum vulgare,Crop,H1P,8


Now, we load the sampling data (S1), and we group all sites / collections (there are date redundancies).

In [3]:
sampling = pd.read_csv("input/mcleish24.TableS1.csv", sep=';')
sampling = sampling.groupby(['Site_code', 'Collection_code', 'Location', 'Longitude', 'Latitude'], as_index = False)['Date'].apply(list)
sampling

,Site_code,Collection_code,Location,Longitude,Latitude,Date
0,C1,C1F,Aranjuez,-3.593308,40.051302,[1/2/16]
1,C1,C3F,Aranjuez,-3.593308,40.051302,[30/1/17]
2,C2,C2F,Aranjuez,-3.599064,40.043193,[1/2/16]
3,C2,C4F,Aranjuez,-3.599064,40.043193,[30/1/17]
4,E1,E1F,Aranjuez,-3.500323,40.059138,"[19/11/15, 10/11/16]"
5,E1,E1P,Aranjuez,-3.500323,40.059138,"[23/5/16, 3/5/17]"
6,E2,E2F,San Martín de la Vega,-3.536191,40.234966,"[19/11/15, 10/11/16]"
7,E2,E2P,San Martín de la Vega,-3.536191,40.234966,"[23/5/16, 10/5/17]"
8,E3,E3F,Ambite,-3.196057,40.089637,"[1/2/16, 10/11/16]"
9,E3,E3P,Ambite,-3.196057,40.089637,"[31/5/16, 3/5/17]"


## Merge

The next piece of code creates an index that joins libraries to collection codes. Some libraries were built out of mixing samples with different collection codes (e.g. spring and winter samples). Therefore, the library-collection code mapping is not unique. 

In [4]:
libraries['Collection_code'] = libraries['Collection_code'].apply(lambda x: x.split("_"))
tmp = libraries.drop_duplicates(subset=['Library_code'])[['Library_code', 'Collection_code']].to_dict(orient='records')
library_collection_code = []
for item in tmp:
    for collection_code in item['Collection_code']:
        library_collection_code.append({"Library_code": item['Library_code'], "Collection_code": collection_code})
library_collection_code = pd.DataFrame.from_records(library_collection_code)
library_collection_code

,Library_code,Collection_code
0,PV001,M1V
1,PV002,M1V
2,PV003,M1V
3,PV003bgi,M1V
4,PV004bgi,M1V
...,...,...
330,PV586,H1P
331,PV587,H1P
332,PV588,H1P
333,PV589,H1P


Now, we merge it with the S1.

In [5]:
library_site = pd.merge(sampling[['Collection_code', 'Site_code']], library_collection_code, on='Collection_code') # type: ignore
library_site


,Collection_code,Site_code,Library_code
0,C1F,C1,PV534
1,C1F,C1,PV535
2,C1F,C1,PV538
3,C1F,C1,PV540
4,C1F,C1,PV544
...,...,...,...
330,Z1V,Z1,PV590
331,Z2V,Z2,PV047
332,Z2V,Z2,PV048
333,Z2V,Z2,PV527


Finally, we merge some other data attributes such as the host taxon, and the number of extracts. We rename it to harmonize it with future analysis.

In [6]:
library_site.value_counts('Library_code')

Library_code
PV578    3
PV570    3
PV582    2
PV566    2
PV562    2
        ..
PV115    1
PV114    1
PV113    1
PV112    1
PV590    1
Name: count, Length: 323, dtype: int64

In [7]:
sites = pd.merge(
    library_site,
    libraries[['Library_code', 'Habitat', 'No_extracts', 'Host_taxon']], on='Library_code', how='left'
)
# sites['Habitat_type'] = sites['Habitat'].map(sites_to_type)
# sites = sites.drop_duplicates(subset=['Site_code'])[['Site_code', 'Longitude', 'Latitude', 'Location', 'Habitat', 'Habitat_type']].to_dict(orient='records')
sites = sites[['Site_code', 'Library_code', 'Habitat', 'No_extracts', 'Host_taxon']]
sites = sites.rename(columns={'Site_code': 'site', 'Library_code': 'library', 'Habitat': 'habitat', 'No_extracts': 'n_extracts', 'Host_taxon':'host_taxon'})
sites = sites.sort_values(by=['site', 'library'])
sites


,site,library,habitat,n_extracts,host_taxon
0,C1,PV534,Crop,3,Diplotaxis erucoides
1,C1,PV535,Crop,17,Brassica oleracea
2,C1,PV538,Crop,8,Brassica oleracea
3,C1,PV540,Crop,1,Picris echioides
4,C1,PV544,Crop,4,Sisymbrium runcinatum
...,...,...,...,...,...
330,Z1,PV590,Crop,11,Zea mays
331,Z2,PV047,Crop,13,Zea mays
332,Z2,PV048,Crop,9,Desconocida 4
333,Z2,PV527,Crop,4,Convolvulus arvensis


## Save

In [8]:
sites.to_csv("output/metadata.site-library.csv", sep=";", index=None) # type: ignore

In [9]:
db.save_dataframe(
    sites, table_name="D_sites", 
    description="This table contains key information about each of the libraries, such as their site, habitat and host"
)

Saved D_sites to db.2025-11-17


In [10]:
db.conn.close()